In [ ]:
import io
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from azure.core.credentials import AzureNamedKeyCredential
from azure.storage.blob import BlobServiceClient


def localizar_raiz_projeto() -> Path:
    diretorio_atual = Path.cwd().resolve()

    for candidato in [
        diretorio_atual,
        *diretorio_atual.parents,
    ]:
        if (
            (candidato / "src").is_dir()
            and (
                candidato
                / "src"
                / "config"
                / "settings.py"
            ).is_file()
        ):
            return candidato

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto. "
        "O arquivo src/config/settings.py não foi encontrado."
    )


ROOT = localizar_raiz_projeto()

if str(ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(ROOT),
    )


IS_DATABRICKS = bool(
    os.getenv("DATABRICKS_RUNTIME_VERSION")
)


if not IS_DATABRICKS:
    from dotenv import load_dotenv

    ENV_PATH = ROOT / ".env"

    if not ENV_PATH.is_file():
        raise FileNotFoundError(
            f"Arquivo .env não encontrado: {ENV_PATH}"
        )

    load_dotenv(
        dotenv_path=ENV_PATH,
        override=True,
    )


from src.config.settings import (
    AZURE_STORAGE_ACCOUNT,
    AZURE_STORAGE_KEY,
    BRONZE_CONTAINER,
    SILVER_PATH,
)


if IS_DATABRICKS:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    spark = (
        SparkSession.builder
        .getOrCreate()
    )


SILVER_META_UF_PATH = (
    SILVER_PATH
    / "meta_uf"
)

SILVER_DQ_PATH = (
    SILVER_PATH
    / "monitoring"
    / "dq_results"
)

SILVER_META_UF_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

SILVER_DQ_PATH.mkdir(
    parents=True,
    exist_ok=True,
)


credential = AzureNamedKeyCredential(
    name=AZURE_STORAGE_ACCOUNT,
    key=AZURE_STORAGE_KEY,
)

account_url = (
    f"https://{AZURE_STORAGE_ACCOUNT}"
    ".blob.core.windows.net"
)

blob_service_client = BlobServiceClient(
    account_url=account_url,
    credential=credential,
)

blob_service_client.get_account_information()

container_client = (
    blob_service_client.get_container_client(
        container=BRONZE_CONTAINER
    )
)

container_client.get_container_properties()


SUFIXO_ARQUIVO = (
    "meta_alfabetizacao_uf.parquet"
)

candidatos = [
    blob.name
    for blob in container_client.list_blobs()
    if blob.name.endswith(SUFIXO_ARQUIVO)
]

if not candidatos:
    raise FileNotFoundError(
        "Nenhum arquivo Bronze encontrado com o sufixo: "
        f"{SUFIXO_ARQUIVO}"
    )

arquivo_mais_recente = max(
    candidatos
)

print(
    f"✅ Ambiente: "
    f"{'Databricks' if IS_DATABRICKS else 'local'}"
)

print(
    f"✅ Lendo Bronze: "
    f"{arquivo_mais_recente}"
)


blob_client = (
    container_client.get_blob_client(
        blob=arquivo_mais_recente
    )
)

conteudo = (
    blob_client
    .download_blob()
    .readall()
)

if not conteudo:
    raise RuntimeError(
        f"O blob está vazio: {arquivo_mais_recente}"
    )

pdf = pd.read_parquet(
    io.BytesIO(conteudo),
    engine="pyarrow",
)

if pdf.empty:
    raise ValueError(
        "O arquivo Bronze produziu um DataFrame vazio."
    )


COLUNAS_OBRIGATORIAS = [
    "ano",
    "sigla_uf",
    "rede",
]

colunas_ausentes = [
    coluna
    for coluna in COLUNAS_OBRIGATORIAS
    if coluna not in pdf.columns
]

if colunas_ausentes:
    raise ValueError(
        "Colunas obrigatórias ausentes no Bronze: "
        f"{colunas_ausentes}. "
        f"Colunas disponíveis: {list(pdf.columns)}"
    )


pdf["ano"] = (
    pdf["ano"]
    .astype("string")
    .str.strip()
)

pdf["sigla_uf"] = (
    pdf["sigla_uf"]
    .astype("string")
    .str.strip()
    .str.upper()
)

pdf["rede"] = (
    pdf["rede"]
    .astype("string")
    .str.strip()
)


rede_map = {
    "2": "Estadual",
    "3": "Municipal",
    "4": "Privada",
}

pdf["rede_nome"] = (
    pdf["rede"]
    .map(rede_map)
    .fillna("Desconhecida")
)


if "_ingested_at" in pdf.columns:
    pdf["ingested_at"] = pd.to_datetime(
        pdf["_ingested_at"],
        errors="coerce",
        utc=True,
    )
else:
    pdf["ingested_at"] = pd.Timestamp.now(
        tz="UTC"
    )


pdf["source"] = (
    f"azure://{BRONZE_CONTAINER}/"
    f"{arquivo_mais_recente}"
)

pdf["version"] = "1.0"

pdf["_silver_processed_at"] = (
    pd.Timestamp.now(
        tz="UTC"
    )
)


for coluna in COLUNAS_OBRIGATORIAS:
    pdf[coluna] = pdf[coluna].replace(
        "",
        pd.NA,
    )


nulos_chave = int(
    pdf[
        COLUNAS_OBRIGATORIAS
    ]
    .isna()
    .sum()
    .sum()
)

duplicados = int(
    pdf.duplicated(
        subset=COLUNAS_OBRIGATORIAS,
        keep=False,
    ).sum()
)

total_registros = int(
    len(pdf)
)

total_ufs = int(
    pdf["sigla_uf"]
    .dropna()
    .nunique()
)


print(
    f"[DQ] Registros: "
    f"{total_registros}"
)

print(
    f"[DQ] Nulos nas chaves: "
    f"{nulos_chave}"
)

print(
    f"[DQ] Registros com chave duplicada: "
    f"{duplicados}"
)

print(
    f"[DQ] UFs encontradas: "
    f"{total_ufs}"
)


run_at = datetime.now(
    timezone.utc
)

registros_dq = [
    {
        "table_name": "silver.meta_uf",
        "rule": "completude_chaves",
        "status": (
            "PASS"
            if nulos_chave == 0
            else "FAIL"
        ),
        "records_checked": total_registros,
        "failures": nulos_chave,
        "run_at": run_at,
    },
    {
        "table_name": "silver.meta_uf",
        "rule": "unicidade_chave_composta",
        "status": (
            "PASS"
            if duplicados == 0
            else "FAIL"
        ),
        "records_checked": total_registros,
        "failures": duplicados,
        "run_at": run_at,
    },
]

pdf_dq = pd.DataFrame(
    registros_dq
)


colunas_tecnicas_remover = [
    coluna
    for coluna in [
        "_ingested_at",
        "_source_table",
    ]
    if coluna in pdf.columns
]

if colunas_tecnicas_remover:
    pdf = pdf.drop(
        columns=colunas_tecnicas_remover
    )


if nulos_chave > 0:
    raise ValueError(
        "A carga Silver foi interrompida porque existem "
        f"{nulos_chave} valores nulos nas chaves."
    )

if duplicados > 0:
    duplicados_df = (
        pdf.loc[
            pdf.duplicated(
                subset=COLUNAS_OBRIGATORIAS,
                keep=False,
            ),
            COLUNAS_OBRIGATORIAS,
        ]
        .sort_values(
            COLUNAS_OBRIGATORIAS
        )
    )

    print(
        "❌ Exemplos de chaves duplicadas:"
    )

    if IS_DATABRICKS:
        display(
            spark.createDataFrame(
                duplicados_df.head(20)
            )
        )
    else:
        display(
            duplicados_df.head(20)
        )

    raise ValueError(
        "A carga Silver foi interrompida porque existem "
        f"{duplicados} registros com chave duplicada."
    )


if IS_DATABRICKS:
    spark.sql(
        "CREATE DATABASE IF NOT EXISTS monitoring"
    )

    spark.sql(
        """
        CREATE TABLE IF NOT EXISTS monitoring.dq_results (
            table_name STRING,
            rule STRING,
            status STRING,
            records_checked BIGINT,
            failures BIGINT,
            run_at TIMESTAMP
        )
        USING DELTA
        """
    )

    df_dq = spark.createDataFrame(
        pdf_dq
    )

    df_dq = df_dq.withColumn(
        "run_at",
        F.to_timestamp(
            F.col("run_at")
        ),
    )

    (
        df_dq.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "monitoring.dq_results"
        )
    )

    spark.sql(
        "CREATE DATABASE IF NOT EXISTS silver"
    )

    df_silver = spark.createDataFrame(
        pdf
    )

    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true",
        )
        .partitionBy(
            "ano"
        )
        .saveAsTable(
            "silver.meta_uf"
        )
    )

    registros_gravados = (
        spark.table(
            "silver.meta_uf"
        )
        .count()
    )

    particoes = (
        spark.table(
            "silver.meta_uf"
        )
        .select(
            "ano"
        )
        .distinct()
        .count()
    )

    print()
    print(
        "✅ Tabela Delta gravada: "
        "silver.meta_uf"
    )

    print(
        "✅ DQ gravado em: "
        "monitoring.dq_results"
    )

    print(
        f"✅ Registros: "
        f"{registros_gravados}"
    )

    print(
        f"✅ Partições por ano: "
        f"{particoes}"
    )

    display(
        spark.table(
            "silver.meta_uf"
        )
        .limit(10)
    )

else:
    arquivo_silver = (
        SILVER_META_UF_PATH
        / "meta_uf.parquet"
    )

    arquivo_dq = (
        SILVER_DQ_PATH
        / "dq_results.parquet"
    )

    pdf.to_parquet(
        arquivo_silver,
        index=False,
        engine="pyarrow",
        partition_cols=[
            "ano",
        ],
    )

    if arquivo_dq.is_file():
        dq_existente = pd.read_parquet(
            arquivo_dq,
            engine="pyarrow",
        )

        pdf_dq_final = pd.concat(
            [
                dq_existente,
                pdf_dq,
            ],
            ignore_index=True,
        )
    else:
        pdf_dq_final = pdf_dq

    pdf_dq_final.to_parquet(
        arquivo_dq,
        index=False,
        engine="pyarrow",
    )

    print()
    print(
        f"✅ Silver local gravada em: "
        f"{arquivo_silver.resolve()}"
    )

    print(
        f"✅ DQ local gravado em: "
        f"{arquivo_dq.resolve()}"
    )

    print(
        f"✅ Registros: "
        f"{len(pdf)}"
    )

    print(
        f"✅ Partições por ano: "
        f"{pdf['ano'].nunique()}"
    )

    display(
        pdf.head(10)
    )


✅ Ambiente: local
✅ Lendo Bronze: 2026-10-18_meta_alfabetizacao_uf.parquet
[DQ] Registros: 81
[DQ] Nulos nas chaves: 0
[DQ] Registros com chave duplicada: 0
[DQ] UFs encontradas: 27

✅ Silver local gravada em: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/silver/meta_uf/meta_uf.parquet
✅ DQ local gravado em: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/silver/monitoring/dq_results/dq_results.parquet
✅ Registros: 81
✅ Partições por ano: 3


,ano,sigla_uf,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,rede_nome,ingested_at,source,version,_silver_processed_at
0,2024,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00
1,2023,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00
2,2024,SE,Pública,38.39,38.3,45.9,53.6,61.2,68.3,74.6,80.0,92.84,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00
3,2023,SE,Pública,31.30,38.3,45.9,53.6,61.2,68.3,74.6,80.0,88.34,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00
4,2025,SE,Pública,50.00,38.0,46.0,54.0,61.0,68.0,75.0,80.0,87.00,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00
5,2025,RN,Pública,48.00,44.0,51.0,57.0,64.0,70.0,75.0,80.0,85.00,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00
6,2025,BA,Pública,55.00,43.0,50.0,57.0,64.0,70.0,75.0,80.0,87.00,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00
7,2023,BA,Pública,36.80,43.4,50.2,57.1,63.6,69.8,75.2,80.0,84.53,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00
8,2024,BA,Pública,35.96,43.4,50.2,57.1,63.6,69.8,75.2,80.0,90.04,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00
9,2024,RN,Pública,39.29,43.8,50.6,57.3,63.8,69.9,75.3,80.0,77.72,Desconhecida,2026-10-18 12:36:27.774760+00:00,azure://bronze/2026-10-18_meta_alfabetizacao_u...,1.0,2026-10-18 16:16:31.692796+00:00


In [ ]:
print("Diretório atual:", Path.cwd())
print("Raiz localizada:", ROOT)
print("src existe:", (ROOT / "src").is_dir())
print(
    "settings existe:",
    (
        ROOT
        / "src"
        / "config"
        / "settings.py"
    ).is_file(),
)
print(
    "Raiz no sys.path:",
    str(ROOT) in sys.path,
)

Diretório atual: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/notebooks/silver
Raiz localizada: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28
src existe: True
settings existe: True
Raiz no sys.path: True
